# 1970 Conway's Game of Life

In 1970, mathematician John Conway introduced the Game of Life, a cellular automaton in which simple local rules create surprisingly complex behavior. Although it is not an AI algorithm by itself, Life became an important model for studying computation, emergence, and how organized behavior can arise without a central controller.

## High-Level Ideas

The world is a grid of cells. Each cell is either **alive** or **dead**, and its next state depends on its eight surrounding neighbors. Every cell updates at the same time using four rules:

1. A live cell with fewer than two live neighbors dies from underpopulation.
2. A live cell with two or three live neighbors survives.
3. A live cell with more than three live neighbors dies from overpopulation.
4. A dead cell with exactly three live neighbors becomes alive.

Repeatedly applying these rules produces stable patterns, oscillators that repeat, and spaceships that move across the grid. The global behavior is *emergent*: it comes from many local interactions rather than an explicit plan.

## Example: Oscillators and Spaceships

The simulation below stores each live cell as an `(x, y)` coordinate. It counts the neighbors around every active area and applies Conway's rules to create the next generation. A **blinker** returns to its starting shape after two generations, while a **glider** returns to its starting shape shifted one cell diagonally after four generations.

In [1]:
from collections import Counter


# These eight offsets describe the Moore neighborhood around one cell.
NEIGHBOR_OFFSETS = [
    (x_offset, y_offset)
    for y_offset in (-1, 0, 1)
    for x_offset in (-1, 0, 1)
    if (x_offset, y_offset) != (0, 0)
]


def next_generation(live_cells):
    # Counting neighbors around live cells also discovers nearby dead birth candidates.
    neighbor_counts = Counter(
        (x + x_offset, y + y_offset)
        for x, y in live_cells
        for x_offset, y_offset in NEIGHBOR_OFFSETS
    )

    # Three neighbors causes birth; two preserves a cell only when it is already alive.
    return {
        cell
        for cell, live_neighbors in neighbor_counts.items()
        if live_neighbors == 3 or (live_neighbors == 2 and cell in live_cells)
    }


# Text rendering keeps the simulation visible without a graphics dependency.
def render(live_cells, width, height):
    return "\n".join(
        "".join("█" if (x, y) in live_cells else "·" for x in range(width))
        for y in range(height)
    )


# A horizontal blinker becomes vertical, then returns to its original state.
blinker = {(1, 2), (2, 2), (3, 2)}
vertical_blinker = next_generation(blinker)
returned_blinker = next_generation(vertical_blinker)

print("Blinker, generation 0")
print(render(blinker, width=5, height=5))
print("\nBlinker, generation 1")
print(render(vertical_blinker, width=5, height=5))

# Verify both halves of the oscillator's two-generation cycle.
assert vertical_blinker == {(2, 1), (2, 2), (2, 3)}
assert returned_blinker == blinker


# A glider repeats its shape after four updates, displaced one cell diagonally.
glider = {(1, 0), (2, 1), (0, 2), (1, 2), (2, 2)}
current_generation = glider

for generation in range(5):
    print(f"\nGlider, generation {generation}")
    print(render(current_generation, width=6, height=6))
    if generation < 4:
        current_generation = next_generation(current_generation)

# Compare coordinates rather than relying only on the printed picture.
shifted_glider = {(x + 1, y + 1) for x, y in glider}
assert current_generation == shifted_glider

print("\nGame of Life checks passed.")

Blinker, generation 0
·····
·····
·███·
·····
·····

Blinker, generation 1
·····
··█··
··█··
··█··
·····

Glider, generation 0
·█····
··█···
███···
······
······
······

Glider, generation 1
······
█·█···
·██···
·█····
······
······

Glider, generation 2
······
··█···
█·█···
·██···
······
······

Glider, generation 3
······
·█····
··██··
·██···
······
······

Glider, generation 4
······
··█···
···█··
·███··
······
······

Game of Life checks passed.


## Example: Interactive Game Window

A small Pygame application lets you edit the grid and watch generations evolve. From the repository root, launch it with:

```sh
poetry run python notebooks/game_of_life_window.py
```

Click cells to toggle them, then use **Start**, **Step**, **Randomize**, or **Clear**. The window starts with a glider and wraps cells around the grid edges. Close the window to stop the command.

## Why This Mattered

Conway's Game of Life showed that a tiny set of deterministic rules can support rich, unpredictable, and even universal computation. It influenced research into artificial life, complex systems, distributed intelligence, and emergent computation—areas that ask how sophisticated behavior can grow from simple interacting parts.